# Initialize

In [ ]:
# Python
import logging, socket, sys, uuid;
import datetime, random, socket, time, uuid, os
from typing import Iterable
from tqdm import tqdm

# OTel Resources
from opentelemetry import baggage, trace
from opentelemetry.sdk.resources import Resource

# OTel Metrics
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.exporter.otlp.proto.http.metric_exporter import OTLPMetricExporter
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader
from opentelemetry.metrics import (
    CallbackOptions,
    Observation,
    get_meter_provider,
    set_meter_provider,
)
from opentelemetry.sdk.metrics._internal.export import (
    AggregationTemporality,
    ConsoleMetricExporter
)

# OTel Tracing
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import (
    ConsoleSpanExporter,
    BatchSpanProcessor
)
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.trace import Status, StatusCode, SpanKind

# OTel Logging
from opentelemetry._logs import set_logger_provider
from opentelemetry.exporter.otlp.proto.http._log_exporter import (
    OTLPLogExporter,
)
from opentelemetry.sdk._logs.export import (
    ConsoleLogExporter,
    BatchLogRecordProcessor,
    SimpleLogRecordProcessor
)
from opentelemetry.sdk._logs import LoggerProvider, LoggingHandler

deploymentEnvironmentName = "otel-adventure"
otlp_traces_endpoint = os.environ['OTEL_EXPORTER_OTLP_TRACES_ENDPOINT']
otlp_metrics_endpoint = os.environ['OTEL_EXPORTER_OTLP_METRICS_ENDPOINT']
otlp_logs_endpoint = os.environ['OTEL_EXPORTER_OTLP_LOGS_ENDPOINT']

resource=Resource.create({
        "service.name": __name__,
        "service.instance.id": str(uuid.uuid4()),
        "deployment.environment.name": deploymentEnvironmentName,
        "host.name": socket.gethostname(),
    })
print(f"resource={resource.__dict__}")

# Confirm settings

In [ ]:
print(f"deploymentEnvironmentName: {deploymentEnvironmentName}")

print(f"OTEL_EXPORTER_OTLP_TRACES_ENDPOINT: {otlp_traces_endpoint}")
print(f"OTEL_EXPORTER_OTLP_TRACES_HEADERS: {os.environ['OTEL_EXPORTER_OTLP_TRACES_HEADERS']}")
print(f"OTEL_EXPORTER_OTLP_METRICS_ENDPOINT: {otlp_metrics_endpoint}")
print(f"OTEL_EXPORTER_OTLP_METRICS_HEADERS: {os.environ['OTEL_EXPORTER_OTLP_METRICS_HEADERS']}")
print(f"OTEL_EXPORTER_OTLP_LOGS_ENDPOINT: {otlp_logs_endpoint}")
print(f"OTEL_EXPORTER_OTLP_LOGS_HEADERS: {os.environ['OTEL_EXPORTER_OTLP_LOGS_HEADERS']}")

# Send some traces

In [ ]:
def getTracer(service_name):
    provider = TracerProvider(resource=resource)
    provider.add_span_processor(BatchSpanProcessor(OTLPSpanExporter(endpoint=otlp_traces_endpoint)))
    print(provider._resource.__dict__)
    return trace.get_tracer("python", tracer_provider=provider)
    
def frontend():
    frontend_tracer = getTracer("frontend")
    with frontend_tracer.start_as_current_span("frontend",kind=SpanKind.SERVER) as frontend_span:
        # frontend_span.set_attribute("business.unit", "BusinessUnitD")

        print("Processing web transaction...") 
        time.sleep(random.random())
        handle_checkout()
        time.sleep(random.random())
    
        frontend_span.set_status(Status(StatusCode.OK))
        print("Transaction complete.")
        trace_id = trace.format_trace_id(frontend_span.context.trace_id)
        print(f"Trace ID: {trace_id}")
        print(f"View trace in Datadog: https://app.datadoghq.com/apm/traces?query=%40otel.trace_id%3A{trace_id}")
        print(f"View flow map: https://app.datadoghq.com/apm/traces?query_a=%40otel.trace_id%3A{trace_id}&view=traces&traceQuery=a")
        return trace_id

def handle_checkout():
    checkout_tracer = getTracer("checkout")
    with checkout_tracer.start_as_current_span("checkout") as checkout_span:
        print("Handling checkout...")
        checkout_span.set_attribute("order_num", int(datetime.datetime.timestamp(datetime.datetime.now())*1000) % 100000)
        time.sleep(random.random())
        handle_payment()
        time.sleep(random.random())
        handle_shipping()
        time.sleep(random.random())
    
        checkout_span.set_status(Status(StatusCode.OK))
        print("Checkout complete.")
        
def handle_payment():
    payment_tracer = getTracer("payment")
    with payment_tracer.start_as_current_span("payment") as payment_span:
        print("Handling payment...")
        payment_span.set_attribute("payment_id", str(uuid.uuid4()))
        if (random.random() < 0.1):
            payment_span.set_status(Status(StatusCode.ERROR))
        else:
            time.sleep(random.random())
            payment_span.set_status(Status(StatusCode.OK))
        print("Payment complete.")
    
def handle_shipping():
    shipping_tracer = getTracer("shipping")
    with shipping_tracer.start_as_current_span("shipping") as shipping_span:
        print("Handling shipping...")
        shipping_span.set_attribute("tracking_num", str(uuid.uuid4()))
        time.sleep(random.random())
        print("Shipping complete.")

frontend()

# Send some metrics

In [ ]:
metric_exporter = OTLPMetricExporter(endpoint=otlp_metrics_endpoint)
reader = PeriodicExportingMetricReader(exporter=metric_exporter, export_interval_millis=5000)

meter_provider = MeterProvider(metric_readers=[reader], resource=resource)
print(meter_provider._sdk_config.resource.__dict__)
set_meter_provider(meter_provider)

In [ ]:
def observable_counter_func(options: CallbackOptions) -> Iterable[Observation]:
    yield Observation(1, {})

def observable_up_down_counter_func(options: CallbackOptions) -> Iterable[Observation]:
    yield Observation(-10, {})

def observable_gauge_func(options: CallbackOptions) -> Iterable[Observation]:
    yield Observation(9, {})

In [ ]:
meter_provider = get_meter_provider().get_meter("getting-started", "0.1.2")

# Counter
counter = meter_provider.create_counter("otel_adventure.counter")
counter.add(1)

# Async Counter
observable_counter = meter_provider.create_observable_counter("otel_adventure.observable_counter", [observable_counter_func])

# UpDownCounter
updown_counter = meter_provider.create_up_down_counter("otel_adventure.updown_counter")
updown_counter.add(1)
updown_counter.add(-5)

# Async UpDownCounter
observable_updown_counter = meter_provider.create_observable_up_down_counter(
    "otel_adventure.observable_updown_counter", [observable_up_down_counter_func]
)

# Histogram
histogram = meter_provider.create_histogram("otel_adventure.histogram")
histogram.record(99.9)

# Async Gauge
gauge = meter_provider.create_observable_gauge("otel_adventure.gauge", [observable_gauge_func])

In [ ]:
for n in tqdm(range(60)):
    updown_counter.add(random.randint(-20, 20))
    time.sleep(1)

In [ ]:
metric_exporter.shutdown()

# Send some logs

In [ ]:
log_provider = LoggerProvider(resource=Resource.create(resource)
print(log_provider._resource.__dict__)

log_exporter = OTLPLogExporter(endpoint=otlp_logs_endpoint)

# FOR LAB EXERCISES: No batching of log events...
log_provider.add_log_record_processor(SimpleLogRecordProcessor(log_exporter))
# log_provider.add_log_record_processor(SimpleLogRecordProcessor(ConsoleLogExporter()))

In [ ]:
# Setup a logger
logger = logging.getLogger()
logger.handlers.clear()
logger.addHandler(logging.StreamHandler(sys.stderr))

# Attach OTLP handler to root logger
handler = LoggingHandler(logger_provider=log_provider)
logger.addHandler(handler)

In [ ]:
logger = logging.getLogger()

# In Python, use setLevel() to enable DEBUG output for the logger instance
logger.setLevel(logging.DEBUG)

id=str(uuid.uuid4())
logger.debug(f"This is a DEBUG message, id:{id}")
logger.info(f"This is an INFO message, id:{id}")
logger.warning(f"This is a WARNING message, id:{id}")
logger.error(f"This is an ERROR message, id:{id}")